# 05 · Embedding distance vs. experimental affinity change

Does a larger shift of the embedding after a mutation go together with a larger change in
interaction strength? For every protein, model and mutated position `p` the Pearson correlation is
computed across the point mutations of that position (report, section 3):

$$ r_p = \mathrm{corr}\big(E_m,\; \Delta Z_m\big) $$

where $E_m$ is the **local** distance $d_p$ (`r_local`) or the **global** distance $D$ (`r_global`)
of mutation $m$, and $\Delta Z_m$ is the experimental ΔZ-score (Pm-Rb construct).

**Inputs:** `data/processed/df_dms.csv`, `df_tp.csv` (notebook 01) and
`data/processed/mutant_embedding_distances.csv` (notebook 03).
**Output:** `results/pearson_by_position.csv` and figures.

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# ---------------------------------------------------------------------------
# Project paths
#   * Locally: start Jupyter from the `notebooks/` folder.
#   * Google Colab: Google Drive is mounted; edit the folder name below so it
#     points to the project folder stored in your Drive.
#   * Anywhere: set the PROJECT_DIR environment variable to override.
# ---------------------------------------------------------------------------
try:
    from google.colab import drive

    drive.mount("/content/drive")
    DEFAULT_PROJECT_DIR = Path("/content/drive/MyDrive/plm-embeddings-lxcxe-prb")
except ImportError:
    DEFAULT_PROJECT_DIR = Path.cwd().parent

PROJECT_DIR = Path(os.environ.get("PROJECT_DIR", DEFAULT_PROJECT_DIR))
RAW_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
RESULTS_DIR = PROJECT_DIR / "results"
FIGURES_DIR = PROJECT_DIR / "figures"
for folder in (PROCESSED_DIR, RESULTS_DIR, FIGURES_DIR):
    folder.mkdir(parents=True, exist_ok=True)


def save_figure(name):
    """Save the current matplotlib figure into figures/ (call before plt.show())."""
    plt.savefig(FIGURES_DIR / name, dpi=300, bbox_inches="tight")

import matplotlib.patches as patches
import seaborn as sns
from scipy.stats import pearsonr

## 1. Load and merge

In [ ]:
df_tp = pd.read_csv(PROCESSED_DIR / "df_tp.csv")
df_dms = pd.read_csv(PROCESSED_DIR / "df_dms.csv")
df_distances = pd.read_csv(PROCESSED_DIR / "mutant_embedding_distances.csv")

PROTEINS = df_tp["protein"].tolist()
MODELS = df_distances["embedding_model"].unique().tolist()
WT_PEPTIDE = df_tp.set_index("protein")["peptide"].to_dict()
CORE_POSITIONS = (5, 7, 9)  # L, C, E of the LxCxE motif
MUTATED_POSITIONS = list(range(2, 16))

# Attach the experimental ΔZ-scores computed in notebook 01
df_result = df_distances.merge(
    df_dms[["protein", "mutation", "delta_pm_rb", "delta_pw_rb"]],
    on=["protein", "mutation"],
    how="left",
    validate="many_to_one",
)
assert df_result["delta_pm_rb"].notna().all(), "Some mutations have no experimental ΔZ"
print(df_result.groupby(["embedding_model", "protein"], sort=False).size().unstack())

## 2. Pearson correlation per position

In [ ]:
def safe_pearson(x, y):
    """Pearson r, or NaN when it is undefined (too few points or constant input)."""
    x, y = np.asarray(x, dtype=float), np.asarray(y, dtype=float)
    if len(x) < 2 or np.std(x) == 0 or np.std(y) == 0:
        return np.nan
    return pearsonr(x, y)[0]


rows = []
for (protein, model, position), group in df_result.groupby(
    ["protein", "embedding_model", "position"], sort=False
):
    rows.append({
        "protein": protein,
        "embedding_model": model,
        "position": position,
        "n_mutations": len(group),
        "r_local": safe_pearson(group["dist_local"], group["delta_pm_rb"]),
        "r_global": safe_pearson(group["dist_global"], group["delta_pm_rb"]),
    })

df_corr = pd.DataFrame(rows)
df_corr.to_csv(RESULTS_DIR / "pearson_by_position.csv", index=False)
df_corr.head()

## 3. Heatmaps per model

Rows: protein × (local, global). Columns: mutated position. Positions 5, 7 and 9 (LxCxE core) are
highlighted (report figures 6, S2 and S3).

In [ ]:
def plot_model_heatmap(model_name, highlight_positions=CORE_POSITIONS, vmin=-1, vmax=1, save=True):
    """Local and global Pearson r for all proteins and positions for one model."""
    sub = df_corr[df_corr["embedding_model"] == model_name]

    rows = {}
    for protein in PROTEINS:
        by_position = sub[sub["protein"] == protein].set_index("position")
        for metric in ("local", "global"):
            rows[f"{protein} (r-{metric})"] = by_position[f"r_{metric}"].reindex(MUTATED_POSITIONS)
    heatmap = pd.DataFrame(rows).T

    plt.figure(figsize=(11, 5.5))
    ax = sns.heatmap(heatmap, cmap="coolwarm", annot=True, fmt=".2f", vmin=vmin, vmax=vmax,
                     linewidths=0.5, cbar_kws={"label": "Pearson r"})
    for idx, (tick, pos) in enumerate(zip(ax.get_xticklabels(), MUTATED_POSITIONS)):
        if pos in highlight_positions:
            tick.set_color("red")
            tick.set_fontweight("bold")
            tick.set_fontsize(13)
            ax.add_patch(patches.Rectangle((idx, 0), 1, len(heatmap), fill=False,
                                           edgecolor="red", lw=2.5, clip_on=False))
    ax.set_yticklabels(heatmap.index, rotation=0)
    ax.set_xlabel("Mutated position")
    ax.set_ylabel("")
    plt.title(f"Local and global Pearson correlation with ΔZ-score ({model_name})",
              fontsize=14, fontweight="bold", pad=15)
    plt.tight_layout()
    if save:
        slug = model_name.split(" ")[0].lower().replace("-", "")
        save_figure(f"pearson_heatmap_{slug}.png")
    plt.show()


for model_name in MODELS:
    plot_model_heatmap(model_name)

## 4. Position profiles per protein

In [ ]:
def plot_position_profiles(protein, save=True):
    """r(position) curves of every model, local (left) and global (right)."""
    sub = df_corr[df_corr["protein"] == protein]
    sequence = WT_PEPTIDE[protein]
    fig, axes = plt.subplots(1, 2, figsize=(16, 4.5), sharey=True)
    for ax, metric in zip(axes, ("r_local", "r_global")):
        sns.lineplot(data=sub, x="position", y=metric, hue="embedding_model",
                     marker="o", linewidth=2, ax=ax)
        ax.axhline(0, color="gray", linestyle="--", linewidth=0.8)
        ax.set_xticks(MUTATED_POSITIONS)
        ax.set_xticklabels([f"{sequence[p - 1]}{p}" for p in MUTATED_POSITIONS])
        for tick, pos in zip(ax.get_xticklabels(), MUTATED_POSITIONS):
            if pos in CORE_POSITIONS:
                tick.set_color("red")
                tick.set_fontweight("bold")
        ax.set_title(f"{metric} — {protein}", fontweight="bold")
        ax.set_xlabel("Residue and position (WT sequence)")
        ax.set_ylabel("Pearson r")
        ax.legend(title="Embedding model")
    plt.tight_layout()
    if save:
        save_figure(f"pearson_profile_{protein}.png")
    plt.show()


for protein in PROTEINS:
    plot_position_profiles(protein)